In [1]:
import pandas as pd
import geopandas as gpd
import sys
import importlib.util
import os


if os.getcwd().endswith("notebooks"):
    lib_path = "../lib"
else:
    lib_path = "lib"

modules = [("lib", "__init__.py")]

for module_name, module_path in modules:
    spec = importlib.util.spec_from_file_location(module_name, os.path.join(lib_path, module_path))
    module_obj = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module_obj
    spec.loader.exec_module(module_obj)

In [3]:
assert criteria == "passenger/hour"
assert scope in ["all_day", "off_peak", "peak"]
assert isinstance(transit_modes, list)

In [4]:
def is_within_peak(start, end):
    off_peak_periods = [(7*3600, 9*3600), (16*3600, 19*3600)]
    for p_start, p_end in off_peak_periods:
        for t in [start, end]:
            if p_start <= t <= p_end:
                return True
    return False

In [5]:
gdf_area = gpd.read_file(area_path)
gdf_lines = gpd.read_file(schedule_gpkg_path)

df_legs = pd.read_csv(pt_legs_path, sep=";").merge(pd.read_csv(legs_path, sep=";"), how="left")

gdf_lines = gdf_lines.sjoin(gdf_area, predicate="within")

if len(transit_modes) > 0:
    df_legs = df_legs[df_legs["transit_mode"].isin(transit_modes)]
    gdf_lines = gdf_lines[gdf_lines["mode"].isin(transit_modes)]

df_legs = df_legs[df_legs["transit_line_id"].isin(gdf_lines["line_id"])]

df_legs["arrival_time"] = df_legs["departure_time"] + df_legs["travel_time"]

df_legs["departure_time"] //= 3600
df_legs["departure_time"] = df_legs["departure_time"].astype("int")
df_legs["arrival_time"] //= 3600
df_legs["arrival_time"] = df_legs["arrival_time"].astype("int")

df_legs["covered_hours"] = df_legs.apply(lambda r: list(range(r["departure_time"], r["arrival_time"]+1, 1)),axis=1)

In [6]:
df_legs = df_legs.explode("covered_hours").rename(columns=dict(covered_hours="covered_hour"))
df_counts = df_legs[["covered_hour", "transit_line_id"]].value_counts()
df_counts.head()

covered_hour  transit_line_id
17            IDFM:C00170        237
8             IDFM:C00170        217
18            IDFM:C00170        216
7             IDFM:C00170        153
16            IDFM:C00170        150
Name: count, dtype: int64

In [7]:
df_legs.head()

,person_id,person_trip_id,leg_index,access_stop_id,egress_stop_id,transit_line_id,transit_route_id,departure_id,access_area_id,egress_area_id,...,departure_time,travel_time,vehicle_distance,routed_distance,mode,euclidean_distance,origin_link_id,destination_link_id,arrival_time,covered_hour
10,12140661,0,1,IDFM:9404.link:336511,IDFM:478611.link:326018,IDFM:C02449,IDFM:FOE:157659-C02449-524-C02449-46554-7650250,IDFM:FOE:157659-C02449-524-C02449-46554-765023...,IDFM:59767,IDFM:59836,...,5,816.0,3452.710640,3452.710640,pt,1387.594222,336511,326018,5,5
12,5193844,0,1,IDFM:478571.link:680978,IDFM:478611.link:326018,IDFM:C02450,IDFM:FOE:157659-C02450-524-C02450-46554-7650297,IDFM:FOE:157659-C02450-524-C02450-46554-765028...,IDFM:427072,IDFM:59836,...,5,644.0,1422.983539,1422.983539,pt,1199.478369,680978,326018,5,5
27,4422008,0,1,IDFM:31767.link:690506,IDFM:424133.link:412131,IDFM:C00170,IDFM:TRANSDEV_SUD_YVELINES:151877-C00170-18942338,IDFM:TRANSDEV_SUD_YVELINES:151877-C00170-18942...,IDFM:59998,IDFM:60665,...,5,2693.0,15434.919044,15434.919044,pt,11092.860405,690506,412131,6,5
27,4422008,0,1,IDFM:31767.link:690506,IDFM:424133.link:412131,IDFM:C00170,IDFM:TRANSDEV_SUD_YVELINES:151877-C00170-18942338,IDFM:TRANSDEV_SUD_YVELINES:151877-C00170-18942...,IDFM:59998,IDFM:60665,...,5,2693.0,15434.919044,15434.919044,pt,11092.860405,690506,412131,6,6
28,4764717,0,1,IDFM:31736.link:94269,IDFM:424133.link:412131,IDFM:C00170,IDFM:TRANSDEV_SUD_YVELINES:151877-C00170-18942338,IDFM:TRANSDEV_SUD_YVELINES:151877-C00170-18942...,IDFM:60021,IDFM:60665,...,4,3769.0,9804.568177,9804.568177,pt,8041.757058,94269,412131,6,4


In [8]:
s = df_counts.reindex(pd.MultiIndex.from_product([list(range(0, max(24, df_legs["covered_hour"].max()) + 1, 1)),
                                                  list(gdf_lines["line_id"].unique())
                                                  ],
                                                 names=["covered_hour", "transit_line_id"]))
assert s.sum() == df_counts.sum()
df_counts /= sampling
df_counts = s.reset_index()

In [9]:
if scope == "off_peak":
    df_counts = df_counts[~df_counts["covered_hour"].apply(lambda h: is_within_peak(h, h))]
elif scope == "peak":
    df_counts = df_counts[df_counts["covered_hour"].apply(lambda h: is_within_peak(h, h))]

In [10]:
df_counts["count"].sum()

np.float64(5363.0)

In [11]:
df_counts = df_counts.groupby("transit_line_id")["count"].max().reset_index()
df_selected = df_counts[df_counts["count"] < threshold]
df_selected[["transit_line_id"]].to_csv(output_path, header=True, index=False)